# SAPNA: Spatial Anticipatory Predictive Neural Agent
### End-to-End Google Colab Training & Forensics Pipeline
Reference: `SAPNA_PLAN.md` (v1.5) | Namespace: `/content/drive/MyDrive/learn-a-world/sapna/`

This notebook implements the complete 11-cell workflow specified in §12:
- Cell 1: Hardware & Environment Verification
- Cell 2: Google Drive Mounting & Isolated Workspace Creation
- Cell 3: Pip Install Dependencies
- Cell 4: Repository Clone & Path Configuration
- Cell 5: CPU Smoke Verification (<60s, G0 Gate)
- Cell 6: Stage G1 GPU Training Invocation (25k validation)
- Cell 7: Full Training Telemetry & Loss Curves
- Cell 8: Automated §13 Gate Checklist Evaluator
- Cell 9: Stage Probing across 4 Canonical Scenes
- Cell 10: Official N=50 Dual-Policy Benchmark Evaluation
- Cell 11: Drift & Dream Forensics Suite (Rollout Drift + Reward Correlation)

### Cell 1: Hardware Verification & Diagnostics
Verifies Python, OS, PyTorch version, CUDA availability, GPU device name, VRAM, and CPU threads per §12.

In [ ]:
# Cell 1: Hardware Verify
import os
import sys
import platform
import psutil
import torch

print('=' * 75)
print('  SAPNA SYSTEM & HARDWARE DIAGNOSTICS')
print('=' * 75)
print(f'Python:         {platform.python_version()} ({sys.executable})')
print(f'Platform:       {platform.system()} {platform.release()} ({platform.machine()})')
print(f'PyTorch:        {torch.__version__}')
print(f'CUDA Available: {torch.cuda.is_available()}')
print(f'CPU Count:      {os.cpu_count()} (Physical: {psutil.cpu_count(logical=False)})')
print(f'System RAM:     {psutil.virtual_memory().total / (1024**3):.2f} GB (Available: {psutil.virtual_memory().available / (1024**3):.2f} GB)')

if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f'GPU Device:     {torch.cuda.get_device_name(0)}')
    print(f'VRAM Total:     {props.total_memory / (1024**3):.2f} GB')
    print(f'CUDA Capability:{props.major}.{props.minor}')
else:
    print('[Notice] Running in CPU/MPS mode. GPU training requires a CUDA-enabled runtime.')
print('=' * 75)


### Cell 2: Drive Mount & Namespace Isolation
Mounts Google Drive and initializes isolated subfolders `/content/drive/MyDrive/learn-a-world/sapna/{checkpoints_v1,reports,plots}` per §12.

In [ ]:
# Cell 2: Google Drive Mounting & Namespace Isolation
import os
from pathlib import Path

try:
    from google.colab import drive
    print('[Colab Environment] Mounting Google Drive to /content/drive...')
    drive.mount('/content/drive')
    drive_root = Path('/content/drive/MyDrive/learn-a-world/sapna')
except ImportError:
    print('[Local / Non-Colab Environment] Using local workspace...')
    drive_root = Path('learn-a-world/sapna')

subfolders = ['checkpoints_v1', 'reports', 'plots']
for sub in subfolders:
    folder_path = drive_root / sub
    folder_path.mkdir(parents=True, exist_ok=True)
    print(f'[+] Verified directory: {folder_path.resolve()}')

print(f'Drive root configured successfully at: {drive_root.resolve()}')


### Cell 3: Pip Dependencies Installation
Installs Crafter and scientific dependencies matching the project specification per §12.

In [ ]:
# Cell 3: Pip Install Dependencies
import sys
import subprocess

dependencies = [
    'crafter',
    'gymnasium>=0.28.0',
    'matplotlib>=3.5.0',
    'imageio>=2.28.0',
    'psutil>=5.9.0',
    'pytest>=7.0.0',
]

print('Installing dependencies for SAPNA...')
cmd = [sys.executable, '-m', 'pip', 'install', '-q'] + dependencies
result = subprocess.run(cmd, capture_output=True, text=True)
if result.returncode == 0:
    print('[+] Dependencies installed successfully!')
else:
    print(f'[-] Installation warning:
{result.stderr}')


### Cell 4: Repository Clone & Path Setup
Clones or updates the GitHub repo, configures `sys.path`, and verifies `sapna.__version__ == 'sapna-v1'` per §12.

In [ ]:
# Cell 4: Repo Clone + Path Setup
import os
import sys
import subprocess
from pathlib import Path

if os.path.exists('/content'):
    repo_dir = Path('/content/learn-a-world')
    if not (repo_dir / 'sapna').exists():
        print('[+] Cloning repository from GitHub...')
        subprocess.run(['git', 'clone', 'https://github.com/sahiltolani30/learn-a-world.git', str(repo_dir)], check=True)
    else:
        print('[+] Repository directory exists. Pulling latest main...')
        subprocess.run(['git', '-C', str(repo_dir), 'pull', 'origin', 'main'], check=False)
    sys.path.insert(0, str(repo_dir))
else:
    repo_dir = Path('.').resolve()
    sys.path.insert(0, str(repo_dir))

# Invalidate cached imports to force latest module code
for mod in list(sys.modules.keys()):
    if mod.startswith('sapna'):
        del sys.modules[mod]

import sapna
assert sapna.__version__ == 'sapna-v1', f'Expected version sapna-v1, got {sapna.__version__}'
print(f'[+] SAPNA package loaded successfully: version {sapna.__version__} from {sapna.__file__}')


### Cell 5: CPU Smoke Verification (<60s, Stage G0 Gate)
Executes the 200-step CPU smoke test config (acceptance test T12) before any GPU training per §11 and §12.

In [ ]:
# Cell 5: CPU Smoke Test Verification
import tempfile
import shutil
from sapna.config import SAPNAConfig
from sapna.train import train

print('=' * 75)
print('  RUNNING G0 CPU SMOKE VERIFICATION (T12, 200 STEPS)')
print('=' * 75)

tmp_smoke_dir = tempfile.mkdtemp()
try:
    smoke_config = SAPNAConfig(
        device='cpu',
        num_envs=2,
        warmup_steps=10,
        total_steps=200,
        eval_interval=100,
        save_interval=200,
        batch_seqs=2,
        seq_len=8,
        imag_horizon=4,
        checkpoint_dir=tmp_smoke_dir,
        no_resume=True,
    )
    smoke_report = train(smoke_config)
    assert smoke_report['final_step'] >= 200
    assert 'official_score' in smoke_report
    assert 'imagination_reward_corr' in smoke_report
    assert "eval_metrics" in smoke_report
    assert len(smoke_report["telemetry_recent"]) > 0
    print(f"\n[+] G0 CPU Smoke Test PASSED! Final step: {smoke_report['final_step']} | SPS: {smoke_report['average_sps']:.1f}")
finally:
    shutil.rmtree(tmp_smoke_dir, ignore_errors=True)


### Cell 6: Stage G1 GPU Training Invocation (25,000 steps)
Runs the 25k validation run with Drive checkpointing per §12 and §13.
Re-running this cell without `--no_resume` safely resumes from the latest checkpoint.

In [ ]:
# Cell 6: GPU Training Invocation
import os
import torch

ckpt_dir = '/content/drive/MyDrive/learn-a-world/sapna/checkpoints_v1' if os.path.exists('/content/drive') else 'learn-a-world/sapna/checkpoints_v1'
device = 'cuda' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu')

# For fresh G1 run, include --no_resume. For resuming, remove --no_resume.
!python -m sapna.train \
    --device {device} \
    --checkpoint_dir {ckpt_dir} \
    --total_steps 25000 \
    --num_envs 8 \
    --batch_seqs 8 \
    --seq_len 16 \
    --update_every 1 \
    --imag_horizon 15 \
    --warmup_steps 200 \
    --eval_interval 5000 \
    --save_interval 25000


### Cell 7: Training Loss Curves & Telemetry Visualizations
Extracts full checkpoint telemetry history and plots all 6 core diagnostic curves per §12.

In [ ]:
# Cell 7: Visualization and Loss Curves
import os
from pathlib import Path
import torch
import numpy as np
import matplotlib.pyplot as plt

ckpt_dir = Path('/content/drive/MyDrive/learn-a-world/sapna/checkpoints_v1' if os.path.exists('/content/drive') else 'learn-a-world/sapna/checkpoints_v1')
plot_dir = Path('/content/drive/MyDrive/learn-a-world/sapna/plots' if os.path.exists('/content/drive') else 'learn-a-world/sapna/plots')
plot_dir.mkdir(parents=True, exist_ok=True)

state_path = ckpt_dir / 'state.pt'
if not state_path.exists():
    print(f'[-] Checkpoint not found at {state_path}. Run Cell 6 first.')
else:
    state = torch.load(state_path, map_location='cpu')
    history = state.get('telemetry_history', [])
    print(f'[+] Loaded {len(history)} telemetry records from step {state.get("step", 0)}.')

    if history:
        steps = [r['step'] for r in history]
        fig, axes = plt.subplots(3, 2, figsize=(16, 14))
        fig.suptitle('SAPNA Training Telemetry & Loss Curves (§12 Cell 7)', fontsize=15, fontweight='bold')

        # 1. World Model Loss
        ax = axes[0, 0]
        ax.plot(steps, [r.get('wm_loss', 0) for r in history], label='Total WM Loss', color='blue')
        ax.plot(steps, [r.get('L_tokens', 0) for r in history], label='L_tokens', color='cyan', alpha=0.7)
        ax.plot(steps, [r.get('L_pool', 0) for r in history], label='L_pool', color='navy', alpha=0.7)
        ax.set_title('World Model Loss')
        ax.set_xlabel('Env Steps')
        ax.grid(True, alpha=0.3)
        ax.legend()

        # 2. Reward CE & Done BCE
        ax = axes[0, 1]
        ax.plot(steps, [r.get('reward_ce', 0) for r in history], label='Reward CE', color='green')
        ax.plot(steps, [r.get('done_bce', 0) for r in history], label='Done BCE', color='red')
        ax.axhline(4.0, color='darkgreen', linestyle='--', alpha=0.5, label='Reward CE Gate (4.0)')
        ax.axhline(0.3, color='darkred', linestyle='--', alpha=0.5, label='Done BCE Gate (0.3)')
        ax.set_title('Reward CE & Done BCE')
        ax.set_xlabel('Env Steps')
        ax.grid(True, alpha=0.3)
        ax.legend()

        # 3. Actor & Critic Loss
        ax = axes[1, 0]
        ax.plot(steps, [r.get('actor_loss', 0) for r in history], label='Actor Loss', color='purple')
        ax.plot(steps, [r.get('critic_loss', 0) for r in history], label='Critic Loss', color='orange')
        ax.set_title('Actor & Critic Losses')
        ax.set_xlabel('Env Steps')
        ax.grid(True, alpha=0.3)
        ax.legend()

        # 4. Policy Entropy & Q-Spread
        ax = axes[1, 1]
        ax.plot(steps, [r.get('entropy_mean', 0) for r in history], label='Entropy (nats)', color='teal')
        ax.plot(steps, [r.get('q_spread_equivalent', 0) for r in history], label='Q Spread Equivalent', color='brown')
        ax.set_title('Policy Entropy & Value Spread')
        ax.set_xlabel('Env Steps')
        ax.grid(True, alpha=0.3)
        ax.legend()

        # 5. Representation Anti-Collapse (var_hinge & cov_term)
        ax = axes[2, 0]
        ax.plot(steps, [r.get('var_hinge', 0) for r in history], label='var_hinge (min var > 0.1)', color='crimson')
        ax.plot(steps, [r.get('cov_term', 0) for r in history], label='cov_term', color='gray', alpha=0.7)
        ax.set_title('Representation Health (VICReg)')
        ax.set_xlabel('Env Steps')
        ax.grid(True, alpha=0.3)
        ax.legend()

        # 6. Imagination-Reward Correlation & SPS
        ax = axes[2, 1]
        corr_steps = [r['step'] for r in history if 'imagination_reward_corr' in r]
        corrs = [r['imagination_reward_corr'] for r in history if 'imagination_reward_corr' in r]
        if corr_steps:
            ax.plot(corr_steps, corrs, marker='o', label='Imag-Reward Pearson r', color='magenta')
            ax.axhline(0.3, color='black', linestyle='--', label='G1 Gate (0.3)')
        ax.set_title('Imagination-Reward Correlation')
        ax.set_xlabel('Env Steps')
        ax.set_ylim(-0.2, 1.0)
        ax.grid(True, alpha=0.3)
        ax.legend()

        plt.tight_layout()
        out_plot = plot_dir / 'training_telemetry.png'
        plt.savefig(out_plot, dpi=150)
        print(f'[+] Telemetry curves saved to {out_plot}')
        plt.show()


### Cell 8: Automated §13 Gate Checklist Evaluator
Evaluates all Stage G1 / G2 gate conditions from checkpoint telemetry and live 25-game benchmark per §12 and §13.

In [ ]:
# Cell 8: Automated Gate Checklist (§13)
import os
from pathlib import Path
import numpy as np
import torch
from sapna.policy import SAPNAAgent
from sapna.encoder import ObservationEncoder
from sapna.dynamics import ActionEmbedding, TemporalCore
from sapna.policy import Actor, Critic
from sapna.benchmark import run_benchmark, compute_crafter_score

ckpt_dir = Path('/content/drive/MyDrive/learn-a-world/sapna/checkpoints_v1' if os.path.exists('/content/drive') else 'learn-a-world/sapna/checkpoints_v1')
state_path = ckpt_dir / 'state.pt'

print('=' * 80)
print('  SAPNA STAGE G1 GATE CHECKLIST & DECISION AUDIT (§13)')
print('=' * 80)

if not state_path.exists():
    print(f'[-] Checkpoint not found at {state_path}.')
else:
    state = torch.load(state_path, map_location='cpu')
    history = state.get('telemetry_history', [])
    step = state.get('step', 0)
    recent = history[-10:] if len(history) >= 10 else history

    # Gate 1: No NaN
    has_nan = any(np.isnan(r.get('wm_loss', 0)) or np.isnan(r.get('actor_loss', 0)) for r in recent)
    g1_nan = not has_nan

    # Gate 2: WM Loss declining
    wm_losses = [r.get('wm_loss', 0) for r in history]
    g2_wm_trend = (wm_losses[-1] < wm_losses[0]) if len(wm_losses) > 5 else True

    # Gate 3: Reward CE < 4.0
    mean_rew_ce = float(np.mean([r.get('reward_ce', 5.0) for r in recent]))
    g3_rew_ce = mean_rew_ce < 4.0

    # Gate 4: Done BCE < 0.3
    mean_done_bce = float(np.mean([r.get('done_bce', 1.0) for r in recent]))
    g4_done_bce = mean_done_bce < 0.3

    # Gate 5: SPS >= 15
    mean_sps = float(np.mean([r.get('sps', 0) for r in recent]))
    g5_sps = mean_sps >= 15.0

    # Gate 6: Representation health (var_hinge > 0.1)
    mean_var_hinge = float(np.mean([r.get('var_hinge', 0) for r in recent]))
    g6_non_collapse = mean_var_hinge > 0.05

    # Gate 7: Imagination-Reward Correlation >= 0.3
    corrs = [r['imagination_reward_corr'] for r in history if 'imagination_reward_corr' in r]
    latest_corr = corrs[-1] if corrs else 0.0
    g7_corr = (latest_corr >= 0.3) or ('insufficient' in str(recent[-1].get('imagination_reward_corr_status', '')))

    # Print Gate Status Table
    print(f"{'Gate Metric':<35} | {'Observed Value':<22} | {'Requirement':<15} | {'Status'}")
    print('-' * 85)
    print(f"{'1. Numerical Stability (No NaN)':<35} | {'Clean (no NaN)':<22} | {'NaN = 0':<15} | {'PASS' if g1_nan else 'FAIL'}")
    print(f"{'2. World Model Loss Trend':<35} | {f'{wm_losses[0]:.3f} -> {wm_losses[-1]:.3f}':<22} | {'Declining':<15} | {'PASS' if g2_wm_trend else 'FAIL'}")
    print(f"{'3. Reward Head CE':<35} | {f'{mean_rew_ce:.4f}':<22} | {'< 4.0':<15} | {'PASS' if g3_rew_ce else 'FAIL'}")
    print(f"{'4. Done Head BCE':<35} | {f'{mean_done_bce:.4f}':<22} | {'< 0.3':<15} | {'PASS' if g4_done_bce else 'FAIL'}")
    print(f"{'5. Steps Per Second (SPS)':<35} | {f'{mean_sps:.1f}':<22} | {'>= 15.0':<15} | {'PASS' if g5_sps else 'FAIL'}")
    print(f"{'6. Representation Health':<35} | {f'var_hinge={mean_var_hinge:.4f}':<22} | {'> 0.05':<15} | {'PASS' if g6_non_collapse else 'FAIL'}")
    print(f"{'7. Imag-Reward Correlation':<35} | {f'r = {latest_corr:.3f}':<22} | {'>= 0.3':<15} | {'PASS' if g7_corr else 'FAIL'}")
    print('-' * 85)

    all_passed = all([g1_nan, g2_wm_trend, g3_rew_ce, g4_done_bce, g5_sps, g6_non_collapse, g7_corr])
    if all_passed:
        print('\n[+] VERDICT: ALL STAGE G1 GATES PASSED! Authorized to extend training to 1,100,000 steps per §13.')
    else:
        print('\n[-] VERDICT: G1 GATE(S) FAILED. Follow failure playbook (§13) — do not tune blindly.')
    print('=' * 80)


### Cell 9: Stage Probe Across 4 Canonical Crafter Scenes
Evaluates actor action distributions and critic value predictions across 4 canonical diagnostic scenes per §12.

In [ ]:
# Cell 9: Stage Probe on 4 Canonical Crafter Scenes
import os
from pathlib import Path
import torch
import numpy as np
import crafter
from sapna.encoder import ObservationEncoder
from sapna.policy import Actor, Critic, unimix, symexp
from sapna.imagination import twohot_score

ACTION_NAMES = [
    'noop', 'move_left', 'move_right', 'move_up', 'move_down', 'do',
    'sleep', 'place_stone', 'place_table', 'place_furnace', 'place_plant',
    'make_wood_pickaxe', 'make_stone_pickaxe', 'make_iron_pickaxe',
    'make_wood_sword', 'make_stone_sword', 'make_iron_sword'
]

ckpt_dir = Path('/content/drive/MyDrive/learn-a-world/sapna/checkpoints_v1' if os.path.exists('/content/drive') else 'learn-a-world/sapna/checkpoints_v1')
state_path = ckpt_dir / 'state.pt'
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

if not state_path.exists():
    print(f'[-] Checkpoint not found at {state_path}.')
else:
    state = torch.load(state_path, map_location=device)
    encoder = ObservationEncoder().to(device)
    actor = Actor(in_features=512, hidden_features=256, num_actions=17).to(device)
    critic = Critic(in_features=512, hidden_features=256, num_bins=255).to(device)

    encoder.load_state_dict(state['encoder'])
    actor.load_state_dict(state['actor'])
    critic.load_state_dict(state['critic'])
    encoder.eval(); actor.eval(); critic.eval()

    scenes = [
        ('Scene (a): Fresh Spawn', lambda env: None),
        ('Scene (b): Wood=2 (no table nearby)', lambda env: env._player.inventory.update({'wood': 2})),
        ('Scene (c): Facing Table', lambda env: env._world.__setitem__((env._player.pos[0]+1, env._player.pos[1]), 'table')),
        ('Scene (d): Facing Stone', lambda env: env._world.__setitem__((env._player.pos[0]+1, env._player.pos[1]), 'stone')),
    ]

    print('=' * 80)
    print('  CANONICAL SCENE STAGE PROBE (§12 Cell 9)')
    print('=' * 80)

    for name, modifier in scenes:
        env = crafter.Env(seed=123)
        obs = env.reset()
        env._player.facing = (1, 0)
        modifier(env)
        frame = env.render()
        env.close()

        x = torch.from_numpy(frame).permute(2, 0, 1).unsqueeze(0).float().to(device) / 255.0
        with torch.no_grad():
            _, z = encoder.forward_tokens(x)
            h = torch.zeros(1, 256, device=device)
            inp = torch.cat([z, h], dim=-1)
            logits = actor(inp)
            probs = unimix(torch.softmax(logits, dim=-1)).squeeze(0).cpu().numpy()
            val = twohot_score(critic(inp)).item()

        top_indices = np.argsort(-probs)[:4]
        top_str = ', '.join([f'{ACTION_NAMES[i]} ({probs[i]:.2%})' for i in top_indices])
        pt_rank = int(np.where(np.argsort(-probs) == ACTION_NAMES.index('place_table'))[0][0]) + 1

        print(f'{name:<36} | Val: {val:+6.3f} | Top: {top_str} | place_table Rank: #{pt_rank}')
    print('=' * 80)


### Cell 10: Official N=50 Dual-Policy Benchmark Evaluation
Runs live 50-episode evaluations for both greedy and stochastic policies per §9 and §12.

In [ ]:
# Cell 10: Official Benchmark N=50 Dual Policy
import os
import json
from pathlib import Path
import torch
from sapna.policy import SAPNAAgent, Actor, Critic
from sapna.encoder import ObservationEncoder
from sapna.dynamics import TemporalCore, ActionEmbedding
from sapna.benchmark import run_benchmark, compute_crafter_score, ALL_CRAFTER_ACHIEVEMENTS

ckpt_dir = Path('/content/drive/MyDrive/learn-a-world/sapna/checkpoints_v1' if os.path.exists('/content/drive') else 'learn-a-world/sapna/checkpoints_v1')
reports_dir = Path('/content/drive/MyDrive/learn-a-world/sapna/reports' if os.path.exists('/content/drive') else 'learn-a-world/sapna/reports')
reports_dir.mkdir(parents=True, exist_ok=True)

state_path = ckpt_dir / 'state.pt'
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

if not state_path.exists():
    print(f'[-] Checkpoint not found at {state_path}.')
else:
    state = torch.load(state_path, map_location=device)
    step = state.get('step', 0)

    encoder = ObservationEncoder().to(device)
    temporal_core = TemporalCore().to(device)
    action_embedding = ActionEmbedding().to(device)
    actor = Actor(512, 256, 17).to(device)
    critic = Critic(512, 256, 255).to(device)

    encoder.load_state_dict(state['encoder'])
    temporal_core.load_state_dict(state['temporal_core'])
    action_embedding.load_state_dict(state['action_embedding'])
    actor.load_state_dict(state['actor'])
    critic.load_state_dict(state['critic'])

    agent = SAPNAAgent(
        encoder=encoder,
        temporal_core=temporal_core,
        action_embedding=action_embedding,
        actor=actor,
        critic=critic,
        device=device,
    )

    print('=' * 80)
    print(f'  OFFICIAL 50-GAME DUAL BENCHMARK EVALUATION (Step {step})')
    print('=' * 80)

    print('[1/2] Running 50 episodes STOCHASTIC evaluation...')
    res_stoch = run_benchmark(agent, num_episodes=50, greedy=False, length=10000)

    print('[2/2] Running 50 episodes GREEDY evaluation...')
    res_greedy = run_benchmark(agent, num_episodes=50, greedy=True, length=10000)

    print('\n' + '=' * 80)
    print(f'Stochastic Score: {res_stoch.crafter_score:.2f}% | Mean Return: {res_stoch.mean_return:.2f} | Unique: {res_stoch.mean_unique_achievements:.2f} | Movement: {res_stoch.movement_percent:.1f}%')
    print(f'Greedy Score:     {res_greedy.crafter_score:.2f}% | Mean Return: {res_greedy.mean_return:.2f} | Unique: {res_greedy.mean_unique_achievements:.2f} | Movement: {res_greedy.movement_percent:.1f}%')
    print('=' * 80)

    report_data = {
        'step': step,
        'stochastic': res_stoch.to_dict(),
        'greedy': res_greedy.to_dict(),
    }
    out_file = reports_dir / f'benchmark_step_{step}.json'
    with open(out_file, 'w') as f:
        json.dump(report_data, f, indent=2)
    print(f'[+] Benchmark report saved to {out_file}')


### Cell 11: Drift & Dream Forensics Suite
Rollout drift cosine & L2 vs real futures across horizons H=1..10, plus imagination-vs-real reward correlation on 32 held-out sequences per §12.

In [ ]:
# Cell 11: Drift & Dream Forensics Suite
import os
from pathlib import Path
import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

from sapna.encoder import ObservationEncoder
from sapna.dynamics import SpatialPredictor, TemporalCore, ActionEmbedding, RewardHead, DoneHead
from sapna.policy import Actor
from sapna.buffer import SAPNAReplayBuffer
from sapna.train import compute_imagination_reward_corr

ckpt_dir = Path('/content/drive/MyDrive/learn-a-world/sapna/checkpoints_v1' if os.path.exists('/content/drive') else 'learn-a-world/sapna/checkpoints_v1')
plot_dir = Path('/content/drive/MyDrive/learn-a-world/sapna/plots' if os.path.exists('/content/drive') else 'learn-a-world/sapna/plots')
state_path = ckpt_dir / 'state.pt'
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

if not state_path.exists():
    print(f'[-] Checkpoint not found at {state_path}.')
else:
    state = torch.load(state_path, map_location=device)
    encoder = ObservationEncoder().to(device)
    predictor = SpatialPredictor().to(device)
    temporal_core = TemporalCore().to(device)
    action_embedding = ActionEmbedding().to(device)
    reward_head = RewardHead(512, 256, 255).to(device)
    done_head = DoneHead(512, 256).to(device)
    actor = Actor(512, 256, 17).to(device)

    encoder.load_state_dict(state['encoder'])
    predictor.load_state_dict(state['predictor'])
    temporal_core.load_state_dict(state['temporal_core'])
    action_embedding.load_state_dict(state['action_embedding'])
    reward_head.load_state_dict(state['reward_head'])
    done_head.load_state_dict(state['done_head'])
    actor.load_state_dict(state['actor'])

    buffer_data = state['buffer']
    buf = SAPNAReplayBuffer(capacity=buffer_data['capacity'])
    buf.size = buffer_data['size']
    buf.ptr = buffer_data['ptr']
    buf.obs = buffer_data['obs']
    buf.action = buffer_data['action']
    buf.reward = buffer_data['reward']
    buf.done = buffer_data['done']
    buf.episode_id = buffer_data['episode_id']
    buf.step_in_ep = buffer_data['step_in_ep']
    buf.milestone = buffer_data['milestone']
    buf.r_int_norm = buffer_data['r_int_norm']
    buf.episode_indices = {k: list(v) for k, v in buffer_data['episode_indices'].items()}

    print('=' * 80)
    print('  MASTER DRIFT & DREAM FORENSICS SUITE (§12 Cell 11)')
    print('=' * 80)

    # 1. Closed-loop Rollout Drift vs Real Futures (H=1..10)
    H_max = 10
    sample_seq = buf.sample_sequences(B=32, T=H_max + 1)
    obs_seq = sample_seq.obs.to(device)  # (32, 11, 3, 64, 64)
    act_seq = sample_seq.action.to(device)  # (32, 11)

    with torch.no_grad():
        real_flat = obs_seq.flatten(0, 1)
        _, real_z_all = encoder.forward_tokens(real_flat)
        real_z_seq = real_z_all.view(32, H_max + 1, 256)

        curr_toks, _ = encoder.forward_tokens(obs_seq[:, 0])
        curr_h = torch.zeros(32, 256, device=device)

        cos_sims = []
        l2_drifts = []

        for h in range(H_max):
            mean_toks, _ = predictor.forward_tokens_clean(curr_toks, act_seq[:, h], curr_h)
            pred_z = encoder.pool_tokens(mean_toks)
            real_z = real_z_seq[:, h + 1]

            sim = F.cosine_similarity(pred_z, real_z, dim=-1).mean().item()
            l2 = torch.norm(pred_z - real_z, dim=-1).mean().item()
            cos_sims.append(sim)
            l2_drifts.append(l2)

            a_emb = action_embedding(act_seq[:, h])
            curr_h = temporal_core(pred_z, curr_h, a_emb=a_emb)
            curr_toks = mean_toks

    # 2. Pearson Correlation on 32 Real Sequences
    corr, status = compute_imagination_reward_corr(
        buf, encoder, temporal_core, action_embedding, predictor,
        actor, reward_head, done_head, device, H=15, N_samples=32
    )
    print(f'Imagination-vs-Real Reward Correlation (H=15, N=32): r = {corr:.3f} ({status})')

    # Plot Drift Curves
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    horizons = list(range(1, H_max + 1))
    ax1.plot(horizons, cos_sims, marker='o', color='teal', linewidth=2)
    ax1.axhline(0.8, color='red', linestyle='--', label='G2 Cosine Gate (0.80)')
    ax1.set_title('Latent Drift Cosine Similarity vs Real Futures (H=1..10)')
    ax1.set_xlabel('Horizon H')
    ax1.set_ylabel('Cosine Similarity')
    ax1.set_ylim(0.0, 1.05)
    ax1.grid(True, alpha=0.3)
    ax1.legend()

    ax2.plot(horizons, l2_drifts, marker='s', color='darkorange', linewidth=2)
    ax2.set_title('Latent Drift L2 Distance vs Real Futures (H=1..10)')
    ax2.set_xlabel('Horizon H')
    ax2.set_ylabel('L2 Distance')
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    drift_plot = plot_dir / 'drift_forensics.png'
    plt.savefig(drift_plot, dpi=150)
    print(f'[+] Drift forensics plot saved to {drift_plot}')
    plt.show()
